CHUẨN BỊ MÔI TRƯỜNG & TẠO THƯ MỤC VOLUME 

In [0]:
TEN = "duykh"
BASE_VOLUME = f"/Volumes/workspace/lab_{TEN}"

# 0. Tạo schema lab_duykh và 2 Volume gốc (raw, _ckpt) trước
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.lab_{TEN}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.lab_{TEN}.raw")
spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.lab_{TEN}._ckpt")

# 1. Tạo các thư mục con bên trong Volume
dbutils.fs.mkdirs(f"{BASE_VOLUME}/raw/landing")
dbutils.fs.mkdirs(f"{BASE_VOLUME}/_ckpt/orders_ckpt")
dbutils.fs.mkdirs(f"{BASE_VOLUME}/_ckpt/orders_schema")

print("Đã khởi tạo thành công cấu trúc thư mục trên Volume:")
print("Landing directory:", f"{BASE_VOLUME}/raw/landing")
print("Checkpoint directory:", f"{BASE_VOLUME}/_ckpt/orders_ckpt")

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/lab_duykh/raw/landing"))


In [0]:
path_landing = "/Volumes/workspace/lab_duykh/raw/landing"
df_batch = spark.read.csv(path_landing, header=True, inferSchema=True)

print("Số dòng đọc batch:", df_batch.count())
df_batch.printSchema()


 DỰNG PIPELINE AUTO LOADER VÀO BRONZE

In [0]:
from pyspark.sql.functions import col, current_timestamp

BASE_DIR   = "/Volumes/workspace/lab_duykh"
LANDING    = f"{BASE_DIR}/raw/landing"
CKPT_DIR   = f"{BASE_DIR}/_ckpt/orders_ckpt"
SCHEMA_DIR = f"{BASE_DIR}/_ckpt/orders_schema"
TARGET_TBL = "workspace.bronze_duykh.raw_orders_duykh"

In [0]:
# 1. Đọc stream với Auto Loader từ Volume nội bộ
df_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("cloudFiles.schemaLocation", SCHEMA_DIR)
         .option("cloudFiles.inferColumnTypes", "true")
         .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Cho phép mở rộng cột
         .option("header", "true")
         .load(LANDING)
)

In [0]:
# 2. Bổ sung các cột metadata kỹ thuật bắt buộc
df_bronze_stream = (
    df_stream
      .withColumn("_ingested_at", current_timestamp())
      .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
# 3. Ghi stream vào bảng Delta với trigger availableNow
query = (
    df_bronze_stream.writeStream
      .format("delta")
      .option("checkpointLocation", CKPT_DIR)
      .option("mergeSchema", "true")
      .trigger(availableNow=True) # Xử lý toàn bộ dữ liệu mới rồi tự ngắt
      .toTable(TARGET_TBL)
)

query.awaitTermination()
print("Pipeline Auto Loader Lần 1 hoàn thành xuất sắc!")

INGEST CƠ SỞ DỮ LIỆU TỪ SUPABASE QUA SPARK JDBC POOLER

In [0]:
from pyspark.sql.functions import current_timestamp, lit

TEN = "duykh"

# 1. Thông số Supabase Pooler của bạn
POOLER_HOST = "aws-0-ap-south-1.pooler.supabase.com"
POOLER_PORT = 5432
DB_NAME     = "postgres"
PROJECT_REF = "zrvziaxxaasagqizgvfd"
DB_USER     = f"postgres.{PROJECT_REF}"
DB_PASS     = "KvGlwPjoh4WFgB9R"

# 2. Xây dựng JDBC URL với mã hóa SSL và prepareThreshold=0
jdbc_url = (
    f"jdbc:postgresql://{POOLER_HOST}:{POOLER_PORT}/{DB_NAME}"
    f"?sslmode=require&prepareThreshold=0"
)

# 3. Đọc dữ liệu qua Spark JDBC thuần
df_supabase = (
    spark.read.format("jdbc")
         .option("url", jdbc_url)
         .option("dbtable", "orders")
         .option("user", DB_USER)
         .option("password", DB_PASS)
         .option("driver", "org.postgresql.Driver")
         .load()
)

print(f"Đã đọc thành công {df_supabase.count()} dòng từ Supabase Pooler!")

# 4. Đảm bảo schema tồn tại và lưu vào Bronze Delta Table
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.bronze_{TEN}")
TARGET_SUPABASE_TABLE = f"workspace.bronze_{TEN}.raw_orders_supabase"

(df_supabase
   .withColumn("_ingested_at", current_timestamp())
   .withColumn("_source_type", lit("SUPABASE_SPARK_JDBC_POOLER"))
   .write
   .format("delta")
   .mode("overwrite")
   .saveAsTable(TARGET_SUPABASE_TABLE))

print(f"Đã lưu bảng: {TARGET_SUPABASE_TABLE}")

In [0]:
%sql
SELECT * FROM workspace.bronze_duykh.raw_orders_supabase LIMIT 5;


XỬ LÝ CDC (CHANGE DATA CAPTURE) TỪ SUPABASE

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# 1. Lấy mốc thời gian lớn nhất hiện tại trong Bronze
watermark_row = spark.sql(f"""
    SELECT COALESCE(MAX(updated_at), '1970-01-01 00:00:00') AS watermark 
    FROM workspace.bronze_{TEN}.raw_orders_supabase
""").collect()[0]["watermark"]

print("Watermark hiện tại:", watermark_row)

# 2. Dùng JDBC subquery để chỉ kéo các bản ghi thay đổi từ Supabase
cdc_subquery = f"(SELECT * FROM orders WHERE updated_at > '{watermark_row}') AS cdc_orders"

df_cdc = (
    spark.read.format("jdbc")
         .option("url", jdbc_url)
         .option("dbtable", cdc_subquery)
         .option("user", DB_USER)
         .option("password", DB_PASS)
         .option("driver", "org.postgresql.Driver")
         .load()
)

cdc_count = df_cdc.count()
print(f"Số bản ghi CDC mới: {cdc_count}")

# 3. Bổ sung 2 cột metadata cho khớp cấu trúc bảng Bronze rồi MERGE INTO
if cdc_count > 0:
    df_cdc_enriched = (
        df_cdc
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_type", lit("SUPABASE_SPARK_JDBC_CDC"))
    )
    
    df_cdc_enriched.createOrReplaceTempView("stg_cdc_updates")
    
    spark.sql(f"""
        MERGE INTO workspace.bronze_{TEN}.raw_orders_supabase AS target
        USING stg_cdc_updates AS source
        ON target.order_id = source.order_id
        WHEN MATCHED THEN
          UPDATE SET *
        WHEN NOT MATCHED THEN
          INSERT *
    """)
    print(f"Đã đồng bộ {cdc_count} thay đổi CDC vào Bronze thành công!")
else:
    print("Không có bản ghi CDC nào mới từ Supabase.")